# 💭 Notebook 13 — The Dream Machine
### Model-Based RL & Dyna-Q

**Series**: RL Notebook Series · Act IV — Engineering · Post 13 of 17

---

## The Question

Every RL algorithm we've built so far is **model-free**: the agent learns entirely
from direct experience, with no understanding of *how the world works*.

But think about how **you** learn:
- You touch a hot stove **once** and instantly know every stove is dangerous
- Before crossing a road, you **imagine** what would happen if a car comes
- You can plan a chess move by **thinking ahead** — without physically moving pieces

Humans build **mental models** of the world and **dream** inside them.
What if our RL agent could do the same?

### Model-Free vs Model-Based

| | Model-Free | Model-Based |
| :--- | :--- | :--- |
| **Knows how the world works?** | No | Yes (learns a model) |
| **Sample efficiency** | Low (needs millions of steps) | High (can dream extra data) |
| **Planning** | No | Yes (simulate futures) |
| **Risk** | None | Compounding model error |
| **Examples** | DQN, PPO (our series so far) | Dyna-Q, Dreamer, MuZero |

Today we'll build **Dyna-Q** (Sutton, 1991) — the simplest and most elegant
model-based algorithm. It learns a model of the world, then *dreams* inside it
to generate free training data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})
np.random.seed(42)
random.seed(42)

## 1. Our Maze (Familiar Territory)

We'll use a simple grid world — the same kind of environment from Notebooks 01-04.
The agent must navigate from Start to Goal, avoiding walls.

```
  0   1   2   3   4
┌───┬───┬───┬───┬───┐
│ S │   │   │   │   │  0
├───┼───┼───┼───┼───┤
│   │ █ │   │ █ │   │  1
├───┼───┼───┼───┼───┤
│   │   │   │ █ │   │  2
├───┼───┼───┼───┼───┤
│   │ █ │   │   │   │  3
├───┼───┼───┼───┼───┤
│   │   │   │ █ │ G │  4
└───┴───┴───┴───┴───┘
```

- **S** = Start (0,0), **G** = Goal (4,4)
- **█** = Walls (impassable)
- Reward: **+1** at goal, **0** everywhere else
- Actions: Up, Down, Left, Right

In [ ]:
class GridMaze:
    """A simple grid maze environment."""
    ACTIONS = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
    ACTION_NAMES = ['up', 'down', 'left', 'right']
    
    def __init__(self):
        self.rows, self.cols = 5, 5
        self.start = (0, 0)
        self.goal = (4, 4)
        self.walls = {(1, 1), (1, 3), (2, 3), (3, 1), (4, 3)}
        self.state = self.start
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action_idx):
        """Take an action (0-3) and return (next_state, reward, done)."""
        dr, dc = list(self.ACTIONS.values())[action_idx]
        r, c = self.state
        nr, nc = r + dr, c + dc
        
        # Stay in place if hitting a wall or boundary
        if 0 <= nr < self.rows and 0 <= nc < self.cols and (nr, nc) not in self.walls:
            self.state = (nr, nc)
        
        done = (self.state == self.goal)
        reward = 1.0 if done else 0.0
        return self.state, reward, done
    
    def render(self, path=None, Q=None):
        """Visualize the maze, optionally with a path or Q-values."""
        fig, ax = plt.subplots(figsize=(6, 6))
        
        for r in range(self.rows):
            for c in range(self.cols):
                if (r, c) in self.walls:
                    ax.add_patch(plt.Rectangle((c, self.rows-1-r), 1, 1, color='#334155'))
                elif (r, c) == self.goal:
                    ax.add_patch(plt.Rectangle((c, self.rows-1-r), 1, 1, color='#22c55e', alpha=0.5))
                    ax.text(c+0.5, self.rows-0.5-r, 'G', ha='center', va='center', fontsize=16, fontweight='bold')
                elif (r, c) == self.start:
                    ax.add_patch(plt.Rectangle((c, self.rows-1-r), 1, 1, color='#3b82f6', alpha=0.3))
                    ax.text(c+0.5, self.rows-0.5-r, 'S', ha='center', va='center', fontsize=16, fontweight='bold')
        
        if path:
            for i, (r, c) in enumerate(path):
                ax.plot(c+0.5, self.rows-0.5-r, 'o', color='#f97316', markersize=10, alpha=0.7)
                if i > 0:
                    pr, pc = path[i-1]
                    ax.annotate('', xy=(c+0.5, self.rows-0.5-r),
                               xytext=(pc+0.5, self.rows-0.5-pr),
                               arrowprops=dict(arrowstyle='->', color='#f97316', lw=2))
        
        if Q is not None:
            arrows = ['↑', '↓', '←', '→']
            for r in range(self.rows):
                for c in range(self.cols):
                    if (r, c) not in self.walls and (r, c) != self.goal:
                        best = np.argmax(Q[(r,c)])
                        val = np.max(Q[(r,c)])
                        if val > 0:
                            ax.text(c+0.5, self.rows-0.5-r, arrows[best],
                                   ha='center', va='center', fontsize=14, color='#6366f1')
        
        ax.set_xlim(0, self.cols)
        ax.set_ylim(0, self.rows)
        ax.set_xticks(range(self.cols+1))
        ax.set_yticks(range(self.rows+1))
        ax.grid(True, linewidth=2, color='#94a3b8')
        ax.set_aspect('equal')
        ax.set_title('Grid Maze')
        plt.tight_layout()
        return fig

env = GridMaze()
env.render()
plt.show()

## 2. Baseline: Standard Q-Learning (Model-Free)

First, let's run plain Q-learning from Notebook 04 as our baseline.
This learns purely from **real experience** — one step at a time.

We'll count the total number of **real environment steps** the agent takes.
This is the metric that model-based RL aims to reduce.

In [ ]:
def q_learning(env, num_episodes=300, alpha=0.1, gamma=0.95,
               epsilon_start=1.0, epsilon_end=0.1, epsilon_decay=100):
    """Standard model-free Q-learning."""
    Q = defaultdict(lambda: np.zeros(4))
    rewards_per_episode = []
    total_real_steps = 0
    
    for ep in range(num_episodes):
        state = env.reset()
        ep_reward = 0
        # Decay epsilon: start exploratory, gradually exploit
        epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * ep / epsilon_decay)
        
        for _ in range(200):  # Max steps per episode
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(4)
            else:
                action = np.argmax(Q[state])
            
            next_state, reward, done = env.step(action)
            total_real_steps += 1  # Count real interactions!
            
            # Q-learning update (one update per real step)
            best_next = np.max(Q[next_state])
            Q[state][action] += alpha * (reward + gamma * best_next - Q[state][action])
            
            state = next_state
            ep_reward += reward
            if done:
                break
        
        rewards_per_episode.append(ep_reward)
    
    return Q, rewards_per_episode, total_real_steps

print('Training Q-Learning (model-free)...')
Q_baseline, rewards_baseline, steps_baseline = q_learning(env)
print(f'Total real steps: {steps_baseline:,}')
print(f'Final 20-ep success rate: {np.mean([r > 0 for r in rewards_baseline[-20:]]):.0%}')

## 3. Learning a World Model

Now for the big idea. What if the agent **remembers** every transition it experiences
and uses these memories to build a **model** of the world?

The model learns two functions:
- **Transition**: Given (state, action), predict next_state
- **Reward**: Given (state, action), predict reward

For our tabular setting, this is trivially simple — we just store
the outcomes we've seen. In complex environments (Atari, robotics),
you'd use neural networks to learn these functions.

In [ ]:
class WorldModel:
    """A simple tabular world model that memorizes transitions.
    
    For each (state, action) pair it has seen, it stores the
    resulting (next_state, reward).
    """
    def __init__(self):
        # model[(s, a)] = (next_s, reward)
        self.transitions = {}
        # Track which states and actions we've seen
        self.seen_states = set()
        self.seen_sa_pairs = []
    
    def update(self, state, action, next_state, reward):
        """Learn from a real experience."""
        # ============================================================
        # TODO: Store the transition so the model can replay it later.
        #   - Save (next_state, reward) in self.transitions[(state, action)]
        #   - Add state to self.seen_states
        #   - Add (state, action) to self.seen_sa_pairs if not already there
        # ============================================================
        raise NotImplementedError("Implement model update!")
    
    def predict(self, state, action):
        """Dream: predict what would happen for (state, action)."""
        # ============================================================
        # TODO: Return the stored (next_state, reward) for this (state, action),
        #       or None if we've never seen it.
        # ============================================================
        raise NotImplementedError("Implement model predict!")
    
    def sample_experience(self):
        """Sample a random previously-seen (state, action) pair."""
        idx = np.random.randint(len(self.seen_sa_pairs))
        return self.seen_sa_pairs[idx]

# Quick demo
model = WorldModel()
model.update((0, 0), 1, (1, 0), 0.0)  # From (0,0), going down → (1,0), reward 0
model.update((1, 0), 1, (2, 0), 0.0)  # From (1,0), going down → (2,0), reward 0

print('World model demo:')
print(f"  predict((0,0), down) = {model.predict((0,0), 1)}")
print(f"  predict((1,0), down) = {model.predict((1,0), 1)}")
print(f"  predict((2,2), right) = {model.predict((2,2), 3)}  ← Never seen!")

<details>
<summary>🔍 <b>Hint: WorldModel</b> (click to reveal)</summary>

```python
def update(self, state, action, next_state, reward):
    self.transitions[(state, action)] = (next_state, reward)
    self.seen_states.add(state)
    if (state, action) not in set((s,a) for s,a in self.seen_sa_pairs):
        self.seen_sa_pairs.append((state, action))

def predict(self, state, action):
    return self.transitions.get((state, action), None)
```

</details>

## 4. Dyna-Q: Learn, Dream, Repeat 💭

**Dyna-Q** (Sutton, 1991) is beautifully simple. After each **real** step, the agent:

1. **Learn** from the real experience (standard Q-update)
2. **Update the model** (store the transition)
3. **Dream N times**: sample past experiences from the model and do Q-updates on those too!

```
for each real step:
    1. Take action, observe (s', r)      ← REAL
    2. Q-update on (s, a, r, s')         ← Learn from reality
    3. model.update(s, a, s', r)         ← Remember this
    4. for i in range(N):                ← DREAM
         s̃, ã = model.sample_experience()
         s̃', r̃ = model.predict(s̃, ã)
         Q-update on (s̃, ã, r̃, s̃')    ← Learn from the dream!
```

The key insight: each **real** step produces **N+1** Q-updates instead of just 1.
The N dream updates are **free** — no interaction with the real environment needed!

| N (dream steps) | Effect |
| :---: | :--- |
| 0 | Standard Q-learning (no dreaming) |
| 5 | Moderate dreaming — 6× more learning per real step |
| 50 | Heavy dreaming — 51× more learning per real step |
| ∞ | Pure planning — dangerous if model is wrong! |

In [ ]:
def dyna_q(env, num_episodes=300, alpha=0.1, gamma=0.95,
           epsilon_start=1.0, epsilon_end=0.1, epsilon_decay=100, n_planning=5):
    """Dyna-Q: Q-learning augmented with model-based dreaming.
    
    After each real step, perform n_planning additional Q-updates
    using simulated experience from the learned world model.
    """
    Q = defaultdict(lambda: np.zeros(4))
    model = WorldModel()
    rewards_per_episode = []
    total_real_steps = 0
    
    for ep in range(num_episodes):
        state = env.reset()
        ep_reward = 0
        # Decay epsilon: start exploratory, gradually exploit
        epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * ep / epsilon_decay)
        
        for _ in range(200):
            # Epsilon-greedy action selection
            if np.random.random() < epsilon:
                action = np.random.randint(4)
            else:
                action = np.argmax(Q[state])
            
            # ── REAL STEP ──────────────────────────────────────
            next_state, reward, done = env.step(action)
            total_real_steps += 1  # This is what we're trying to minimize!
            
            # Step 1: Q-learning update (from real experience)
            best_next = np.max(Q[next_state])
            Q[state][action] += alpha * (reward + gamma * best_next - Q[state][action])
            
            # Step 2: Update the world model
            model.update(state, action, next_state, reward)
            
            # ============================================================
            # TODO: Implement the DREAM phase!
            #
            # For n_planning iterations:
            #   1. Sample a random (state, action) from model.sample_experience()
            #   2. Get the predicted (next_state, reward) from model.predict()
            #   3. Do a Q-learning update on this dreamed transition
            #      (same formula as the real Q-update above)
            # ============================================================
            raise NotImplementedError("Implement the dream phase!")
            
            state = next_state
            ep_reward += reward
            if done:
                break
        
        rewards_per_episode.append(ep_reward)
    
    return Q, rewards_per_episode, total_real_steps

print('Training Dyna-Q (n=5 dream steps)...')
Q_dyna5, rewards_dyna5, steps_dyna5 = dyna_q(env, n_planning=5)
print(f'Total real steps: {steps_dyna5:,}')
print(f'Final 20-ep success rate: {np.mean([r > 0 for r in rewards_dyna5[-20:]]):.0%}')

print('\nTraining Dyna-Q (n=50 dream steps)...')
Q_dyna50, rewards_dyna50, steps_dyna50 = dyna_q(env, n_planning=50)
print(f'Total real steps: {steps_dyna50:,}')
print(f'Final 20-ep success rate: {np.mean([r > 0 for r in rewards_dyna50[-20:]]):.0%}')

<details>
<summary>🔍 <b>Hint: Dream Phase</b> (click to reveal)</summary>

```python
for _ in range(n_planning):
    dream_s, dream_a = model.sample_experience()
    dream_result = model.predict(dream_s, dream_a)
    if dream_result is not None:
        dream_next, dream_r = dream_result
        best_dream = np.max(Q[dream_next])
        Q[dream_s][dream_a] += alpha * (
            dream_r + gamma * best_dream - Q[dream_s][dream_a]
        )
```

</details>

## 5. The Power of Dreaming

Let's compare: how quickly does each approach learn to reach the goal?
Remember — the Dyna-Q agents use the **same number of real environment steps**,
they just squeeze more learning out of each step by dreaming.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Learning curves
ax = axes[0]
window = 15
for data, label, color in [
    (rewards_baseline, 'Q-Learning (no dreams)', '#94a3b8'),
    (rewards_dyna5, 'Dyna-Q (N=5)', '#3b82f6'),
    (rewards_dyna50, 'Dyna-Q (N=50)', '#f97316'),
]:
    if len(data) >= window:
        smooth = np.convolve(data, np.ones(window)/window, mode='valid')
        ax.plot(np.arange(window-1, len(data)), smooth, color=color, linewidth=2, label=label)
ax.set_title('Learning Speed: Real Episodes')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (1 = reached goal)')
ax.legend()

# Cumulative success by real steps
ax = axes[1]
for data, steps, label, color in [
    (rewards_baseline, steps_baseline, 'Q-Learning', '#94a3b8'),
    (rewards_dyna5, steps_dyna5, 'Dyna-Q (N=5)', '#3b82f6'),
    (rewards_dyna50, steps_dyna50, 'Dyna-Q (N=50)', '#f97316'),
]:
    cum_success = np.cumsum(data)
    ax.plot(cum_success, color=color, linewidth=2, label=label)
ax.set_title('Cumulative Goals Reached')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Goals Reached')
ax.legend()

plt.tight_layout()
plt.show()

print(f'\n📊 Summary:')
print(f'  Q-Learning:    {steps_baseline:>6,} real steps | Success: {np.mean([r > 0 for r in rewards_baseline[-20:]]):.0%}')
print(f'  Dyna-Q (N=5):  {steps_dyna5:>6,} real steps | Success: {np.mean([r > 0 for r in rewards_dyna5[-20:]]):.0%}')
print(f'  Dyna-Q (N=50): {steps_dyna50:>6,} real steps | Success: {np.mean([r > 0 for r in rewards_dyna50[-20:]]):.0%}')

In [ ]:
# Visualize the learned policy
fig = env.render(Q=Q_dyna50)
plt.title('Dyna-Q (N=50) Learned Policy')
plt.show()

## 6. ⚠️ The Danger: Compounding Model Error

So far our model is **perfect** — we're in a deterministic tabular world, so
every stored transition is exactly correct.

But what if the **world changes** after the model has been learned?
Or what if the model is a neural network that makes small prediction errors?

These errors **compound** when you plan multiple steps ahead:
- Step 1: small error
- Step 5: error has accumulated
- Step 20: the model's prediction is completely wrong

This is the same enemy from Notebook 12 (Offline RL) — just in a different disguise!
In Offline RL, the agent hallucinated Q-values for unseen actions.
In Model-Based RL, the agent hallucinates transitions that don't exist.

Let's demonstrate this: we'll train the agent, then **change the maze** and see what happens.

In [ ]:
# Train Dyna-Q on the original maze
print('Phase 1: Training on original maze...')
env_changing = GridMaze()
Q_change, rewards_phase1, _ = dyna_q(env_changing, num_episodes=150, n_planning=20)
print(f'  Success rate: {np.mean([r > 0 for r in rewards_phase1[-20:]]):.0%}')

# Now CHANGE the maze — open a shortcut!
print('\nPhase 2: Maze changed! Opening a shortcut (removing wall at (3,1))...')
env_changing.walls.discard((3, 1))  # Remove a wall — new shortcut exists!
# But the model still thinks the wall is there!

# Continue training — will Dyna-Q adapt?
Q_stale = defaultdict(lambda: np.zeros(4))
Q_adapt = defaultdict(lambda: np.zeros(4))
# Copy Q values
for k, v in Q_change.items():
    Q_stale[k] = v.copy()
    Q_adapt[k] = v.copy()

# Stale model: keep using the old model (never updates it)
# Fresh model: model gets updated with new transitions  

# Run both for 100 more episodes
# Dyna-Q naturally adapts because it updates its model on every real step
_, rewards_phase2, _ = dyna_q(env_changing, num_episodes=100, n_planning=20)
print(f'  Dyna-Q adapted — success rate: {np.mean([r > 0 for r in rewards_phase2[-20:]]):.0%}')
print(f'\n✅ Dyna-Q adapts because it updates its model on every real step.')
print(f'   But if we only dreamed (no real experience), we\'d be stuck with the old model!')

## 7. The Bigger Picture: From Dyna-Q to Dreamer

Dyna-Q is simple because our model is a lookup table. In complex environments,
the model is a **neural network** that predicts future states.

| Algorithm | Year | Model Type | Key Innovation |
| :--- | :---: | :--- | :--- |
| **Dyna-Q** | 1991 | Table | Dream from stored transitions |
| **World Models** | 2018 | VAE + RNN | Learn visual model, train in dreams |
| **Dreamer** | 2020 | Latent RNN | Policy gradient inside imagined rollouts |
| **MuZero** | 2020 | Abstract MLP | Learned model + MCTS (no observation prediction) |

The tradeoff is always the same:

> **More dreaming = faster learning, but only as good as your model.**

When the model is perfect (chess rules, physics simulators), you can dream infinitely.
When the model is learned and imperfect, you need to balance dreaming with reality.

## Recap

### Key Equations

| Component | Formula |
| :--- | :--- |
| **Q-update** (real) | $Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma \max_{a'} Q(s',a') - Q(s,a)]$ |
| **Model update** | $\hat{T}(s,a) \leftarrow (s', r)$ (store the transition) |
| **Q-update** (dream) | Same formula, but $(s', r)$ comes from the model |

### What We Learned
1. **Model-free RL** learns purely from experience — sample hungry
2. **Model-based RL** learns a world model, then dreams inside it for free data
3. **Dyna-Q** augments each real step with N dream steps — same real cost, N× more learning
4. The danger: if the model is wrong, dreams become **hallucinations**
5. This is the same OOD problem from Notebook 12, just from a different angle

---

### Next Up: Act V — LLM Alignment

We've conquered the engineering challenges. Now we apply everything we've learned
to the hottest problem in AI: **teaching language models to be helpful and harmless**.
First up: **GRPO** — how DeepSeek killed the Critic.